# TruthGuard — Notebook 4/5
## Étape 4 : Vectorisation, Modèles Baseline, Avancés & Stacking Ensemble

> **Prérequis** : `X_train_text`, `X_test_text`, `y_train`, `y_test` disponibles (issus du Notebook 3).

## 13. Vectorisation TF-IDF (word + char n-grams)

In [ ]:
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

word_tfidf = TfidfVectorizer(
    max_features=10_000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2
)
X_train_word = word_tfidf.fit_transform(X_train_text)
X_test_word  = word_tfidf.transform(X_test_text)

char_tfidf = TfidfVectorizer(
    analyzer="char_wb",
    max_features=5_000,
    ngram_range=(3, 5),
    sublinear_tf=True,
    min_df=2
)
X_train_char = char_tfidf.fit_transform(X_train_text)
X_test_char  = char_tfidf.transform(X_test_text)

X_train_tfidf = hstack([X_train_word, X_train_char])
X_test_tfidf  = hstack([X_test_word,  X_test_char])

print(f"TF-IDF (word+char) — Train : {X_train_tfidf.shape} | Test : {X_test_tfidf.shape}")

## 14. Embeddings sémantiques (SentenceTransformer)

In [ ]:
from sentence_transformers import SentenceTransformer
from pathlib import Path

CACHE_DIR   = Path("/content/drive/MyDrive/cache_data")
EMBED_CACHE = CACHE_DIR / "embeddings_all.npy"

df_indexed = df.reset_index(drop=True)
train_idx  = X_train_text.index
test_idx   = X_test_text.index

if EMBED_CACHE.exists():
    print(" Chargement des embeddings depuis le cache...")
    X_embeddings_all = np.load(EMBED_CACHE)
else:
    model_st = SentenceTransformer("all-MiniLM-L6-v2")
    X_embeddings_all = model_st.encode(
        df_indexed["statement"].fillna("").tolist(),
        show_progress_bar=True,
        batch_size=64
    )
    np.save(EMBED_CACHE, X_embeddings_all)
    print(f" Embeddings sauvegardés : {EMBED_CACHE}")

X_train_emb = X_embeddings_all[train_idx]
X_test_emb  = X_embeddings_all[test_idx]
print(f"Embeddings — Train : {X_train_emb.shape} | Test : {X_test_emb.shape}")

## 15. Features combinées TF-IDF + Embeddings

In [ ]:
X_train_full = hstack([X_train_tfidf, csr_matrix(X_train_emb)])
X_test_full  = hstack([X_test_tfidf,  csr_matrix(X_test_emb)])

print(f"Features combinées (TF-IDF + Embeddings)")
print(f"  Train : {X_train_full.shape}")
print(f"  Test  : {X_test_full.shape}")

FEATURE_SETS = {
    "TF-IDF seul":         (X_train_tfidf, X_test_tfidf),
    "Embeddings seul":     (csr_matrix(X_train_emb), csr_matrix(X_test_emb)),
    "TF-IDF + Embeddings": (X_train_full,  X_test_full),
}

## 16. Modèles baseline

In [ ]:
import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, average_precision_score,
                              confusion_matrix)

ALL_RESULTS  = []
ALL_PROBAS   = {}

def evaluate_model(model, X_tr, y_tr, X_te, y_te,
                   model_name="Model", feature_set="TF-IDF + Embeddings",
                   show_cm=True, store=True):
    t0 = time.time()
    model.fit(X_tr, y_tr)
    train_time = time.time() - t0

    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    results = {
        "Model":        model_name,
        "Features":     feature_set,
        "Accuracy":     accuracy_score(y_te, y_pred),
        "Precision":    precision_score(y_te, y_pred, zero_division=0),
        "Recall":       recall_score(y_te, y_pred, zero_division=0),
        "F1":           f1_score(y_te, y_pred, zero_division=0),
        "AUC-ROC":      roc_auc_score(y_te, y_proba),
        "AUC-PR":       average_precision_score(y_te, y_proba),
        "Train time(s)":round(train_time, 2),
    }

    print(f"\n── {model_name} [{feature_set}] ──")
    for k, v in results.items():
        if k not in ["Model", "Features"]:
            print(f"  {k:16s}: {v:.4f}" if isinstance(v, float) else f"  {k:16s}: {v}")

    if show_cm:
        cm = confusion_matrix(y_te, y_pred)
        fig, ax = plt.subplots(figsize=(4, 3.5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Fake","Real"], yticklabels=["Fake","Real"], ax=ax)
        ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
        ax.set_title(f"Confusion Matrix — {model_name}")
        plt.tight_layout(); plt.show()

    if store:
        ALL_RESULTS.append(results)
        ALL_PROBAS[model_name] = y_proba

    return model, results, y_proba

print(" Fonction evaluate_model prête.")

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB

lr_model, lr_res, y_proba_lr = evaluate_model(
    LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0, random_state=SEED),
    X_train_full, y_train, X_test_full, y_test,
    model_name="Logistic Regression"
)

svm_model, svm_res, y_proba_svm = evaluate_model(
    CalibratedClassifierCV(
        LinearSVC(class_weight="balanced", C=1.0, random_state=SEED), cv=5
    ),
    X_train_full, y_train, X_test_full, y_test,
    model_name="SVM (calibré)"
)

rf_model, rf_res, y_proba_rf = evaluate_model(
    RandomForestClassifier(
        n_estimators=200, class_weight="balanced",
        max_depth=None, min_samples_leaf=2, random_state=SEED, n_jobs=-1
    ),
    X_train_full, y_train, X_test_full, y_test,
    model_name="Random Forest"
)

cnb_model, cnb_res, y_proba_cnb = evaluate_model(
    CalibratedClassifierCV(ComplementNB(alpha=0.1), cv=5),
    X_train_tfidf, y_train, X_test_tfidf, y_test,
    model_name="Complement Naive Bayes",
    feature_set="TF-IDF seul"
)

In [ ]:
print("\n═══ Comparaison des représentations (LR) ═══")
repr_results = []
for feat_name, (X_tr, X_te) in FEATURE_SETS.items():
    m = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
    _, res, _ = evaluate_model(m, X_tr, y_train, X_te, y_test,
                               model_name="LR", feature_set=feat_name,
                               show_cm=False, store=False)
    repr_results.append(res)

repr_df = pd.DataFrame(repr_results).set_index("Features")
print(repr_df[["Accuracy","F1","AUC-ROC"]].round(4))

fig, ax = plt.subplots(figsize=(8, 4))
repr_df[["Accuracy","F1","AUC-ROC"]].plot(kind="bar", ax=ax,
    color=["#3498db","#e74c3c","#2ecc71"], edgecolor="white")
ax.set_title("Impact de la représentation sur LR", fontweight="bold")
ax.set_ylabel("Score")
ax.set_xticklabels(ax.get_xticklabels(), rotation=10)
ax.set_ylim(0.7, 1.02)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 17. Modèles avancés (XGBoost, LightGBM)

In [ ]:
try:
    import xgboost as xgb
    import lightgbm as lgb
    HAS_BOOST = True
except ImportError:
    HAS_BOOST = False

if HAS_BOOST:
    neg = (y_train == 0).sum()
    pos = (y_train == 1).sum()
    spw = neg / pos

    xgb_model, xgb_res, y_proba_xgb = evaluate_model(
        xgb.XGBClassifier(
            n_estimators=300, learning_rate=0.05, max_depth=6,
            scale_pos_weight=spw, use_label_encoder=False,
            eval_metric="logloss", random_state=SEED, n_jobs=-1, verbosity=0
        ),
        X_train_full, y_train, X_test_full, y_test,
        model_name="XGBoost"
    )

    lgb_model, lgb_res, y_proba_lgb = evaluate_model(
        lgb.LGBMClassifier(
            n_estimators=300, learning_rate=0.05, num_leaves=63,
            scale_pos_weight=spw, random_state=SEED, n_jobs=-1, verbose=-1
        ),
        X_train_full, y_train, X_test_full, y_test,
        model_name="LightGBM"
    )
else:
    print(" XGBoost/LightGBM non disponibles, section ignorée.")

## 18. Stacking Ensemble — TruthGuard

In [ ]:
from sklearn.model_selection import StratifiedKFold

def truthguard_stacking(base_models, meta_model, X_tr, y_tr, X_te, y_te,
                        n_splits=5, model_names=None):
    skf      = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    n_base   = len(base_models)
    names    = model_names or [f"Base_{i}" for i in range(n_base)]

    train_meta      = np.zeros((X_tr.shape[0], n_base))
    test_meta_folds = np.zeros((n_splits, X_te.shape[0], n_base))

    for fold_i, (tr_idx, val_idx) in enumerate(skf.split(X_tr, y_tr)):
        X_fold_tr  = X_tr[tr_idx]
        X_fold_val = X_tr[val_idx]
        y_fold_tr  = y_tr.iloc[tr_idx]

        for j, (model, name) in enumerate(zip(base_models, names)):
            model.fit(X_fold_tr, y_fold_tr)
            train_meta[val_idx, j]       = model.predict_proba(X_fold_val)[:, 1]
            test_meta_folds[fold_i, :, j]= model.predict_proba(X_te)[:, 1]

        print(f"  Fold {fold_i+1}/{n_splits} terminé.", end="\r")

    test_meta = test_meta_folds.mean(axis=0)
    meta_model.fit(train_meta, y_tr)
    y_pred  = meta_model.predict(test_meta)
    y_proba = meta_model.predict_proba(test_meta)[:, 1]

    results = {
        "Model":     "TruthGuard (Stacking)",
        "Features":  "TF-IDF + Embeddings",
        "Accuracy":  accuracy_score(y_te, y_pred),
        "Precision": precision_score(y_te, y_pred, zero_division=0),
        "Recall":    recall_score(y_te, y_pred, zero_division=0),
        "F1":        f1_score(y_te, y_pred, zero_division=0),
        "AUC-ROC":   roc_auc_score(y_te, y_proba),
        "AUC-PR":    average_precision_score(y_te, y_proba),
        "Train time(s)": 0,
    }

    print("\n═══ TruthGuard (Stacking) ═══")
    for k, v in results.items():
        if k not in ["Model", "Features", "Train time(s)"]:
            print(f"  {k:16s}: {v:.4f}")

    fig, ax = plt.subplots(figsize=(4.5, 3.5))
    sns.heatmap(confusion_matrix(y_te, y_pred), annot=True, fmt="d", cmap="Purples",
                xticklabels=["Fake","Real"], yticklabels=["Fake","Real"], ax=ax)
    ax.set_xlabel("Prédit"); ax.set_ylabel("Réel")
    ax.set_title("TruthGuard — Matrice de confusion")
    plt.tight_layout(); plt.show()

    return meta_model, results, y_proba, train_meta, test_meta


base_lr  = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
base_rf  = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                   random_state=SEED, n_jobs=-1)
base_svm = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", random_state=SEED), cv=3
)
meta_lr  = LogisticRegression(class_weight="balanced", random_state=SEED)

print(" Entraînement du stacking (5 folds × 3 modèles)...")
tg_meta, tg_res, y_proba_tg, train_meta, test_meta = truthguard_stacking(
    [base_lr, base_rf, base_svm], meta_lr,
    X_train_full, y_train, X_test_full, y_test,
    model_names=["LR", "RF", "SVM"]
)
ALL_RESULTS.append(tg_res)
ALL_PROBAS["TruthGuard (Stacking)"] = y_proba_tg

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
base_names = ["LR", "RF", "SVM"]

for i, (ax, name) in enumerate(zip(axes, base_names)):
    for lbl, color, lab in [(0, "#e74c3c", "Fake"), (1, "#2980b9", "Real")]:
        mask = y_test.values == lbl
        ax.hist(test_meta[mask, i], bins=30, alpha=0.6, color=color, label=lab, density=True)
    ax.set_title(f"Méta-feature : {name}", fontweight="bold")
    ax.set_xlabel("Probabilité prédite")
    ax.legend()

plt.suptitle("Distribution des méta-features (OOF probas)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 19. Comparaison globale des modèles

In [ ]:
results_df = pd.DataFrame(ALL_RESULTS)
print(results_df[["Model","Accuracy","Precision","Recall","F1","AUC-ROC","AUC-PR"]]
      .round(4).to_string(index=False))

In [ ]:
df_cmp = results_df[results_df["Features"] == "TF-IDF + Embeddings"].set_index("Model")

metrics  = ["Accuracy","Precision","Recall","F1","AUC-ROC","AUC-PR"]
x        = np.arange(len(df_cmp))
width    = 0.13
palette  = ["#3498db","#e74c3c","#2ecc71","#f39c12","#9b59b6","#1abc9c"]

fig, ax = plt.subplots(figsize=(15, 6))
for i, (metric, color) in enumerate(zip(metrics, palette)):
    bars = ax.bar(x + (i - 2.5) * width, df_cmp[metric], width,
                  label=metric, color=color, alpha=0.88, edgecolor="white")
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels(df_cmp.index, rotation=12, fontsize=10)
ax.set_ylim(0.70, 1.03)
ax.set_ylabel("Score")
ax.set_title("Comparaison globale des modèles (TF-IDF + Embeddings)",
             fontsize=14, fontweight="bold")
ax.legend(ncol=3, loc="lower right", fontsize=9)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
heat_data = df_cmp[metrics].T
sns.heatmap(heat_data, annot=True, fmt=".4f", cmap="YlGnBu",
            vmin=0.7, vmax=1.0, linewidths=0.5, ax=ax)
ax.set_title("Heatmap des métriques par modèle", fontweight="bold")
plt.tight_layout()
plt.show()